In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, json, os, warnings
warnings.filterwarnings('ignore')

lr = joblib.load('../models/logreg_woe.pkl')
binning_process = joblib.load('../models/binning_process.pkl')

# with open('../models/feature_names.json') as f:
#     features = json.load(f)

train = pd.read_parquet('../data/processed/train_woe.parquet')
oot   = pd.read_parquet('../data/processed/oot_woe.parquet')

X_train = train.drop(columns=['TARGET'])
y_train = train['TARGET']
X_oot   = oot.drop(columns=['TARGET'])
y_oot   = oot['TARGET']

features = X_train.columns.tolist()

# print(f"Features: {len(features)}")

print(f"LR intercept: {lr.intercept_[0]:.4f}")



LR intercept: 0.0730


In [2]:
# 评分卡缩放核心自定义参数（风控行业通用标准）
# PDO=20：分数每提升20分，好坏客户赔率翻倍
# BASE_SCORE=600：基准分数线
# BASE_ODDS=19：基准分600分时，好坏赔率19:1（19个好人，1个逾期坏人）


PDO        = 20
# PDO        = 15  # 备选参数，用于对照实验

BASE_SCORE = 600
BASE_ODDS  = 19   # 基准分数对应的好坏赔率

FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)

print(f"Scaling parameters:")
print(f"  FACTOR = {FACTOR:.4f}")
print(f"  OFFSET = {OFFSET:.4f}")

# 评分卡基础计算公式：模型 logit 是坏/好赔率，评分使用好/坏赔率
# bad_log_odds = intercept + sum(coef_i * WoE_i)
# Score = OFFSET - FACTOR * bad_log_odds
#       = OFFSET - FACTOR * intercept - FACTOR * sum(coef_i * WoE_i)
# 单个分箱得分公式：将 OFFSET 与截距平均分摊到每个特征
# Score per bin = -FACTOR * coef_i * WoE_i + (OFFSET - FACTOR * intercept) / n_features



Scaling parameters:
  FACTOR = 28.8539
  OFFSET = 515.0414


In [3]:
def build_scorecard(lr, binning_process, features, factor, offset):
    """
    将WOE逻辑回归系数与特征分箱WOE信息，转换为标准评分卡表格
    输出每个特征各分箱对应的分值
    """
    n = len(features)
    # 将模型全局截距分摊到所有特征，用于分箱得分计算
    intercept_contribution = (offset - factor * lr.intercept_[0]) / n

    scorecard_rows = []

    for i, feature in enumerate(features):
        coef = lr.coef_[0][i]

        optb = binning_process.get_binned_variable(feature)
        bt = optb.binning_table.build()

        # 剔除汇总行：缺失值、特殊值、总计行
        bt = bt[~bt.index.isin(['Special', 'Missing', 'Totals'])]

        for _, row in bt.iterrows():
            woe = row['WoE']
            # 评分卡分箱分数计算公式
            score = round(-factor * coef * woe + intercept_contribution)

            scorecard_rows.append({
                'Feature': feature,
                'Bin'    : row.name,
                'WoE'    : round(woe, 4),
                'Coefficient': round(coef, 4),
                'Points' : int(score),
            })

    return pd.DataFrame(scorecard_rows)

scorecard = build_scorecard(lr, binning_process, features, FACTOR, OFFSET)
print(f"Scorecard shape: {scorecard.shape}")
scorecard.head(20)



Scorecard shape: (460, 5)


,Feature,Bin,WoE,Coefficient,Points
0,EXT_SOURCE_3,0,-1.1628,-0.6265,-12
1,EXT_SOURCE_3,1,-0.6348,-0.6265,-2
2,EXT_SOURCE_3,2,-0.2238,-0.6265,5
3,EXT_SOURCE_3,3,0.0145,-0.6265,9
4,EXT_SOURCE_3,4,0.1438,-0.6265,12
5,EXT_SOURCE_3,5,0.3473,-0.6265,15
6,EXT_SOURCE_3,6,0.5479,-0.6265,19
7,EXT_SOURCE_3,7,0.5955,-0.6265,20
8,EXT_SOURCE_3,8,0.7335,-0.6265,22
9,EXT_SOURCE_3,9,0.9040,-0.6265,26


In [4]:
# 筛选区分能力最强的前6个特征，绘制各分箱对应分值横向条形图
# 区分能力衡量标准：同一特征最高分 - 最低分，分值跨度越大，风险区分效果越好
top_features = (scorecard.groupby('Feature')['Points']
                .apply(lambda x: x.max() - x.min())
                .sort_values(ascending=False)
                .head(6).index.tolist())

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, feat in zip(axes.flatten(), top_features):
    df_feat = scorecard[scorecard['Feature'] == feat]
    # 分值大于0绿色，小于0红色，直观展示正负贡献
    colors = ['#2ecc71' if p > 0 else '#e74c3c' for p in df_feat['Points']]
    ax.barh(df_feat['Bin'].astype(str), df_feat['Points'], color=colors)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('Points')

plt.suptitle('Scorecard Points by Bin – Top 6 Features', fontsize=13)
plt.tight_layout()
plt.show()



In [5]:
def apply_scorecard(X_woe, scorecard, features, lr, factor, offset):
    """
    批量计算样本信用总分
    原理：累加每个特征分箱对应的分值，得到客户最终评分
    """
    n = len(features)
    # 将模型截距项分摊到各个特征，参与分值计算
    intercept_contribution = (offset - factor * lr.intercept_[0]) / n
    scores = pd.Series(0.0, index=X_woe.index)

    for i, feature in enumerate(features):
        coef = lr.coef_[0][i]
        woe_values = X_woe[feature]
        # 单特征得分计算公式
        points = -factor * coef * woe_values + intercept_contribution
        scores += points

    return scores.round().astype(int)

train_scores = apply_scorecard(X_train, scorecard, features, lr, FACTOR, OFFSET)
oot_scores  = apply_scorecard(X_oot,  scorecard, features, lr, FACTOR, OFFSET)

# 一致性校验：评分必须严格等于 OFFSET - FACTOR * 模型坏账logit（最终统一取整）
train_logit = lr.intercept_[0] + X_train.to_numpy() @ lr.coef_[0]
oot_logit = lr.intercept_[0] + X_oot.to_numpy() @ lr.coef_[0]
expected_train_scores = pd.Series(
    np.rint(OFFSET - FACTOR * train_logit).astype(int), index=X_train.index
)
expected_oot_scores = pd.Series(
    np.rint(OFFSET - FACTOR * oot_logit).astype(int), index=X_oot.index
)
assert train_scores.equals(expected_train_scores)
assert oot_scores.equals(expected_oot_scores)
print('Score formula check passed: Score = OFFSET - FACTOR * bad_log_odds')

print(f"Train scores - min: {train_scores.min()}, max: {train_scores.max()}, mean: {train_scores.mean():.0f}")
print(f"OOT scores   - min: {oot_scores.min()}, max: {oot_scores.max()}, mean: {oot_scores.mean():.0f}")



Score formula check passed: Score = OFFSET - FACTOR * bad_log_odds
Train scores - min: 404, max: 643, mean: 527
OOT scores   - min: 417, max: 642, mean: 527


In [6]:
# PDO=20 时
print("PDO=20, 客户0的分数:", train_scores.iloc[0])

PDO=20, 客户0的分数: 537


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (scores, y, label) in zip(axes, [
    (train_scores, y_train, 'Train'),
    (oot_scores,   y_oot,   'Holdout'),
]):
    for target, color, name in [(0, '#2ecc71', 'Good (no default)'),
                                (1, '#e74c3c', 'Bad (default)')]:
        # density=True 使用密度，消除训练集/留出样本量差异带来的视觉干扰
        ax.hist(scores[y == target], bins=40, alpha=0.6,
                color=color, label=name, density=True)

    ax.set_title(f'Score Distribution – {label}')
    ax.set_xlabel('Scorecard Points')
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.show()



In [8]:
print(f"训练集(Train)标准差: {np.std(train_scores):.4f}")
print(f"验证集(OOT)标准差: {np.std(oot_scores):.4f}")

训练集(Train)标准差: 29.5887
验证集(OOT)标准差: 29.3871


In [9]:
# 将评分十分位分箱，统计每一档逾期率，检验评分卡单调性（风控核心校验）
score_df = pd.DataFrame({
    'score': train_scores,
    'target': y_train
})

# 按客户数量均分，把分数划分成10组（十分位分箱），处理重复分数导致的分箱冲突
score_df['score_bin'] = pd.qcut(score_df['score'], q=10, duplicates='drop')
# 按分数分箱聚合：样本总数、坏账数量、逾期率(坏账率)
dr_by_bin = score_df.groupby('score_bin', observed=True).agg(
    count = ('target', 'count'),
    bad   = ('target', 'sum'),
    dr    = ('target', 'mean')
).reset_index()


fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(range(len(dr_by_bin)), dr_by_bin['count'], color='steelblue', alpha=0.6, label='Count')
ax1.set_xlabel('Score Decile (low → high score)')
ax1.set_ylabel('Count', color='steelblue')


ax2 = ax1.twinx()
ax2.plot(range(len(dr_by_bin)), dr_by_bin['dr'] * 100,
         color='red', marker='o', linewidth=2, label='Default Rate %')
ax2.set_ylabel('Default Rate %', color='red')

ax1.set_title('Default Rate by Score Decile\n(monotonic decrease = good scorecard)')
plt.tight_layout()
plt.show()



In [10]:
# 定义评分卡风险分层阈值（cutoff），输出业务审批策略；同时打印训练集分数各分位数，辅助阈值划定
cutoffs = pd.DataFrame({
    'Risk Band'    : ['Very High', 'High', 'Medium', 'Low', 'Very Low'],
    'Score Range'  : ['< 520', '520–560', '560–600', '600–640', '> 640'],
    'Action'       : ['Reject', 'Manual Review', 'Conditional Approve',
                      'Approve', 'Approve + Best Rate'],
})
print(cutoffs.to_string(index=False))

print("\nScore percentiles (train):")
# 计算训练集分数10~90分位数，用来参考、校验上面人工设定的分层阈值是否合理
for p in [10, 20, 30, 40, 50, 60, 70, 80, 90]:
    print(f" {p}th percentile: {np.percentile(train_scores, p):.0f}")



Risk Band Score Range              Action
Very High       < 520              Reject
     High     520–560       Manual Review
   Medium     560–600 Conditional Approve
      Low     600–640             Approve
 Very Low       > 640 Approve + Best Rate

Score percentiles (train):
 10th percentile: 488
 20th percentile: 502
 30th percentile: 512
 40th percentile: 520
 50th percentile: 528
 60th percentile: 536
 70th percentile: 543
 80th percentile: 552
 90th percentile: 565


In [11]:
os.makedirs('../models', exist_ok=True)

scorecard.to_csv('../models/scorecard_table.csv', index=False)
print("Saved: models/scorecard_table.csv")

pd.DataFrame({'score': train_scores, 'target': y_train}).to_parquet(
    '../data/processed/train_scores.parquet', index=False
)
pd.DataFrame({'score': oot_scores, 'target': y_oot}).to_parquet(
    '../data/processed/oot_scores.parquet', index=False
)
print("Saved: train_scores.parquet, oot_scores.parquet")



Saved: models/scorecard_table.csv
Saved: train_scores.parquet, oot_scores.parquet


## Scorecard Summary

### Scaling Parameters
| Parameter | Value | Meaning |
|---|---|---|
| PDO | 20 | Every 20 points → odds of default halve |
| Base Score | 600 | Score at 1:19 default odds |
| Factor | 'Accept' / 'Reject' | Scaling multiplier |



### Risk Bands
| Risk Band | Score Range | Action |
|---|---|---|
| Very High | < 520 | Reject |
| High | 520–560 | Manual Review |
| Medium | 560–600 | Conditional Approve |
| Low | 600–640 | Approve |
| Very Low | > 640 | Approve + Best Rate |

### Key Observations
- Score distribution shows clear separation between Good and Bad clients
- Default rate decreases monotonically across score deciles — confirms 
  scorecard validity
- Score range of [вставь] points provides sufficient granularity for 
  risk segmentation

**Next step:** Full validation → notebook 08
